### Evaluation: Numerical Drift vs. Inherent Architecture Error (Baseline Spatial UNet)

**Objective:** 
To isolate and quantify the exact cause of error in the baseline NeuralMAG UNet model, distinguishing between its inherent mathematical mapping limitations and the accumulated temporal drift caused by its Markovian (memoryless) architecture.

**Methodology (Clean History Mapping):**
This script initializes a random $64 \times 64$ magnetization state and relaxes it to a 5-vortex ground state. It then executes two parallel LLG simulations:
1. **Film 1 (Ground Truth):** Driven exclusively by the exact Fast Fourier Transform (FFT) solver.
2. **Film 2 (Surrogate):** Driven by the UNet's predicted demagnetizing field.

At every time step, we track two distinct error metrics:
*   **Instantaneous Mapping Error (Blue Line):** We feed the UNet the *clean, perfect history* from Film 1 and compare its prediction to the FFT. This proves the inherent mathematical accuracy of the architecture.
*   **Accumulated Drift (Red Line):** We track the error of Film 2 as the UNet feeds its own corrupted predictions back into itself over thousands of steps. 

**Expected Result:** 
The gap between the Instantaneous Error and the Accumulated Drift mathematically justifies the necessity of upgrading to a non-Markovian (Derivative Buffer) architecture to stabilize long-term temporal predictions.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%cd /content/drive/MyDrive/NeuralMAG_Data/egs/NMI/vortex_evaluate

/content/drive/MyDrive/NeuralMAG_Data/egs/NMI/vortex_evaluate


In [ ]:
!CUDA_VISIBLE_DEVICES=0 python compute_vortex.py \
  --gpu 0 \
  --krn 16 \
  --w 64 \
  --layers 2 \
  --InitCore 5 \
  --ckpt_path "/content/drive/MyDrive/NeuralMAG_Data/NeuralMAG_Models/model_origin.pt"

In [4]:
import sys
import os

# Append the root directory where the 'libs' folder lives
# Update this path if 'libs' is inside a different project root folder
project_root = "/content/drive/MyDrive/NeuralMAG_Data"
if project_root not in sys.path:
    sys.path.append(project_root)

print("Python search path updated successfully!")

Python search path updated successfully!


In [7]:
!cp /content/drive/MyDrive/NeuralMAG_Data/egs/demo/ckpt/k16/model.pt ./model.pt

# For using model with pretrained weights from authors
!PYTHONPATH=/content/drive/MyDrive/NeuralMAG_Data CUDA_VISIBLE_DEVICES=0 python compute_vortex.py \
  --gpu 0 \
  --krn 16 \
  --w 64 \
  --layers 2 \
  --InitCore 5 \
  --ckpt_path "/content/drive/MyDrive/NeuralMAG_Data/egs/demo/ckpt/k16/model.pt"

MAG2305 version: UnetHd_Public_2024.10.17

usage: compute_vortex.py [-h] [--gpu GPU] [--krn KRN] [--w W]
                         [--layers LAYERS] [--split SPLIT]
                         [--InitCore INITCORE] [--modelshape MODELSHAPE]
                         [--Ms MS] [--Ax AX] [--Ku KU] [--Kvec KVEC]
                         [--damping DAMPING] [--Hext_val HEXT_VAL]
                         [--Hext_vec HEXT_VEC] [--dtime DTIME]
                         [--error_min ERROR_MIN] [--max_iter MAX_ITER]
                         [--nsave NSAVE] [--nplot NPLOT] [--nsamples NSAMPLES]
compute_vortex.py: error: unrecognized arguments: --ckpt_path /content/drive/MyDrive/NeuralMAG_Data/egs/demo/ckpt/k16/model.pt


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Load the accumulated drift array generated by compute_vortex.py
drift_error = np.load('/content/drive/MyDrive/NeuralMAG_Data/egs/demo/train/error_film2_drift.npy')
inst_error = np.load('/content/drive/MyDrive/NeuralMAG_Data/egs/demo/train/error_film1_inst.npy')

# Plot the comparison
plt.figure(figsize=(10, 5))
plt.plot(drift_error, color='red', label='Accumulated Drift (film2 - corrupted history feedback)')
plt.plot(inst_error, color='blue', label='Instantaneous Mapping (film1 - clean FFT history)')
plt.title('Numerical Drift vs. Inherent Architecture Error (Baseline UNet)')
plt.xlabel('Simulation Steps')
plt.ylabel('Error (Relative to FFT Ground Truth)')
plt.legend()
plt.grid(True)
plt.show()